# CS769 - Experiment 5 (Kaggle): How many post-merge preference examples repair an alignment-degrading merge?

**Settings:** Accelerator = *GPU T4 x2*, Internet = *On*, Secret `GITHUB_TOKEN` attached (only for the last cell).

**Protocol (plan E5 / Fig. 4)**
1. Take the two-model E2 merges (E_math + E_pref) that degrade alignment most: the **worst** and the **median** PRR case among Linear / TIES / DARE-TIES at 0 < α < 1 (picked automatically from `outputs/results/e2_sweep_results.json`; the merges are rebuilt with E2's own `merge_state_dict`).
2. Repair each with post-merge **DPO on 128 / 512 / 2000 pairs** (≈ plan's 125 / 500 / 2k; multiples of the 8-pair accumulation). Subsets are *nested* (first n of one fixed permutation of the 2,000-pair pool). Same hyper-parameters as E4 (β 0.1, lr 5e-5, cosine, LoRA r16/α32, seed 42); DPO reference = the pre-repair merge.
3. Measure **PRR vs the parent E_pref** (preference margin on 500 held-out pairs, B0-relative), the fraction of the lost gain recovered, SVAMP / CR_math (optionally MBPP + IFEval), with 95% bootstrap CIs.
4. **Control:** DPO from B0 with the same 128 / 512 pairs (is a merged model easier to repair than B0 is to align?).
5. Report the **smallest budget reaching 90% / 95% of the parent PRR** (point estimate, CI lower bound, log-linear interpolation) and **Figure 4**.

Everything is resumable: re-run the same cells after a restart and finished jobs are skipped.

In [ ]:
!nvidia-smi -L

In [ ]:
!pip install -q -U peft datasets accelerate matplotlib
!pip uninstall -y torchao
import transformers, peft; print(transformers.__version__, peft.__version__)

In [ ]:
import os
REPO = "/kaggle/working/LLM_Merging"
os.chdir(REPO)
need = ["src/e5/e5_pipeline.py", "src/e4/e4_pipeline.py", "src/merge/run_e2_sweep.py",
        "src/eval/eval_math.py", "src/train/train_dpo_stable.py", "src/data/precompute_ref_logps.py", "configs/base_config.py",
        "outputs/results/e2_sweep_results.json", "outputs/datasets/e_pref_train.jsonl", "outputs/datasets/e_pref_test.jsonl"] + \
       [f"outputs/adapters/e_{n}/adapter_config.json" for n in ["math", "pref"]]
print("missing:", [p for p in need if not os.path.exists(p)] or "none")
print("E4 outputs present (anchors reusable):", os.path.exists("outputs/e4/results/eval_e_pref.json"),
      "| B0 500-pair cache:", os.path.exists("outputs/e4/b0_pref_500_cache.json"))

## Configuration (edit, then run - the pipeline reads `outputs/e5/config.json`)

In [ ]:
import json, os
CONFIG = {
    "budgets": [128, 512, 2000],        # add 32, 64 for finer resolution; 8000 needs a bigger pool (set pool_path)
    "cases": ["worst", "median"],       # or explicit, e.g. [["linear", 0.1], ["ties", 0.5]]
    "ref_modes": ["pre_dpo"],           # add "b0" for the DPO reference-model ablation (doubles the repair runs)
    "controls": True,                   # DPO-from-B0 control for budgets < 2000
    "full_eval": False,                 # False: SVAMP + preference (fast). True: + MBPP + IFEval (about 3x slower)
    "pool_path": "outputs/datasets/e_pref_train.jsonl",
    "seed": 42,                         # change for extra seeds (separate markers / checkpoints / adapters)
    "thresholds": [0.90, 0.95],
    "reuse_e4_anchors": True            # reuse B0 / E_math / E_pref evals from E4 if present (same hardware)
}
os.makedirs("outputs/e5", exist_ok=True)
json.dump(CONFIG, open("outputs/e5/config.json", "w"), indent=1)
print(json.dumps(CONFIG, indent=1))

In [ ]:
import subprocess, sys, queue, concurrent.futures as cf
N_GPU = max(1, len(subprocess.run("nvidia-smi -L", shell=True, capture_output=True, text=True).stdout.strip().splitlines()))
PIPE = "src/e5/e5_pipeline.py"
def stage_jobs(stage):
    return subprocess.run([sys.executable, PIPE, "list", "--stage", stage], capture_output=True, text=True).stdout.split()
def _run(job, gpu, extra):
    env = dict(os.environ, CUDA_VISIBLE_DEVICES=str(gpu))
    os.makedirs("outputs/e5/logs", exist_ok=True)
    with open(f"outputs/e5/logs/{job}.log", "a") as lg:
        return subprocess.run([sys.executable, PIPE, "run", "--job", job] + extra, env=env, stdout=lg, stderr=subprocess.STDOUT).returncode
def run_queue(jobs, extra=()):
    gpus = queue.Queue()
    for g in range(N_GPU): gpus.put(g)
    def worker(job):
        g = gpus.get()
        try: rc = _run(job, g, list(extra))
        finally: gpus.put(g)
        print(("OK   " if rc == 0 else "FAIL ") + job, flush=True)
        return job, rc
    with cf.ThreadPoolExecutor(N_GPU) as ex: res = list(ex.map(worker, jobs))
    failed = [j for j, rc in res if rc]
    for j in failed:
        print("\n--- log tail:", j); print(open(f"outputs/e5/logs/{j}.log").read()[-2000:])
    return failed
def status(): print(subprocess.run([sys.executable, PIPE, "status"], capture_output=True, text=True).stdout)
print("GPUs:", N_GPU)

## Stage 0 - pick the cases, build the nested subsets, bake E_math / E_pref, B0 reference cache, anchors

In [ ]:
assert not run_queue(["select_cases"])
print(open("outputs/e5/cases.json").read())
assert not run_queue(stage_jobs("setup"))
assert not run_queue(stage_jobs("anchors"))     # reuses E4 results when available, else evaluates B0 / E_math / E_pref
status()

## Stage 1 - rebuild the chosen E2 merges (CPU) and precompute DPO reference log-probs (GPU)

In [ ]:
assert not run_queue(stage_jobs("merge"))
assert not run_queue(stage_jobs("ref"))

## Stage 2 - repair DPO runs (+ DPO-from-B0 controls). 8.7 s/step measured on T4 in E4; about 330 steps per case

In [ ]:
fails = run_queue(stage_jobs("repair"))
assert not fails, fails

## Stage 3 - evaluation (`--delete_after` frees checkpoints once nothing needs them)

In [ ]:
fails = run_queue(stage_jobs("eval"), extra=["--delete_after"])
status()

## Stage 4 - Table E5, Figure 4, download pack

In [ ]:
print(subprocess.run([sys.executable, PIPE, "run", "--job", "report"], capture_output=True, text=True).stdout)
print(subprocess.run([sys.executable, PIPE, "run", "--job", "pack"], capture_output=True, text=True).stdout)
from IPython.display import Image, display, FileLink
display(Image("outputs/e5/results/figure4_repair_curve.png")); display(FileLink("e5_artifacts.zip"))

## Push results to GitHub (branch `e5-results`; no checkpoints / temporary files)

In [ ]:
from kaggle_secrets import UserSecretsClient
TOKEN = UserSecretsClient().get_secret("GITHUB_TOKEN")
REPO_URL = "github.com/<your-user>/<your-repo>.git"      # <-- edit
os.chdir("/kaggle/working/LLM_Merging")
def sh(cmd):
    r = subprocess.run(cmd, shell=True, capture_output=True, text=True)
    print((r.stdout + r.stderr).replace(TOKEN, "***")); return r.returncode
sh('git config user.email "you@example.com" && git config user.name "Your Name"')
sh('git checkout -B e5-results')
with open(".gitignore", "a") as f:
    f.write("\noutputs/checkpoints/\noutputs/e5/data/ref_*.jsonl\noutputs/e5/data/subsets/\ne5_artifacts.zip\n")
sh('du -sh outputs/e5')
sh('git add .gitignore outputs/e5 && git status --short | head -40')
sh('git diff --cached --name-only | grep -E "checkpoints|ref_|\\.zip" || echo "no heavy files staged"')
sh('git commit -m "E5 repair-budget results from Kaggle run"')
sh(f'git push https://x-access-token:{TOKEN}@{REPO_URL} e5-results')